In [1]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
import gc

# 데이터 불러오기

In [2]:
# 경로 설정
file_path_d = r"C:\Users\DME_Lab\DME Dropbox\장성우\Business Process Mining"

In [3]:
# 저장된 데이터 불러오기
df_granted = pd.read_csv(
    os.path.join(file_path_d, 'data', 'G02_granted_cases.tsv'),
    sep='\t'
)

C:\Users\DME_Lab\AppData\Local\Temp\ipykernel_42528\2632189921.py:2: DtypeWarning: Columns (6,7) have mixed types. Specify dtype option on import or set low_memory=False.
  df_granted = pd.read_csv(


In [4]:
patent = pd.read_csv(r"C:\Users\DME_Lab\Desktop\DME\WTC\data\original\g_patent.tsv", sep='\t')
inventor = pd.read_csv(r"C:\Users\DME_Lab\Desktop\DME\WTC\data\original\g_inventor_disambiguated.tsv", sep='\t')

C:\Users\DME_Lab\AppData\Local\Temp\ipykernel_42528\343776094.py:1: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  patent = pd.read_csv(r"C:\Users\DME_Lab\Desktop\DME\WTC\data\original\g_patent.tsv", sep='\t')


In [5]:
# intercase features
examiner_workload = pd.read_csv(
    os.path.join(file_path_d, 'data', 'G03_examiner_workload.tsv'),
    sep='\t'
)

artunit_backlog = pd.read_csv(
    os.path.join(file_path_d, 'data', 'G04_artunit_backlog.tsv'),
    sep='\t'
)
def convert_art_unit(x):
    try:
        return str(int(float(x)))
    except:
        return str(x)

artunit_backlog['examiner_art_unit'] = artunit_backlog['examiner_art_unit'].apply(convert_art_unit)

C:\Users\DME_Lab\AppData\Local\Temp\ipykernel_42528\4058900230.py:7: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  artunit_backlog = pd.read_csv(


# 특허데이터 전처리

In [6]:
# inventor 수 집계
patent['patent_id'] = patent['patent_id'].astype(str).str.strip()
inventor['patent_id'] = inventor['patent_id'].astype(str).str.strip()

inventor_count = (inventor.groupby('patent_id')['inventor_sequence']
                  .count()
                  .reset_index()
                  .rename(columns={'inventor_sequence': 'inventor_count'}))

patent_features = patent[['patent_id', 'patent_type', 'num_claims']].merge(
    inventor_count,
    on='patent_id',
    how='left'
)

print(patent_features.shape)
print(patent_features.head())

(8980130, 4)
  patent_id patent_type  num_claims  inventor_count
0  10000000     utility          20             1.0
1  10000001     utility          12             2.0
2  10000002     utility           9             4.0
3  10000003     utility          18             3.0
4  10000004     utility           6             2.0


In [7]:
# str으로 통일
df_granted['patent_number'] = df_granted['patent_number'].astype(str).str.strip()

# patent_features merge
df_granted = df_granted.merge(
    patent_features,
    left_on='patent_number',
    right_on='patent_id',
    how='left'
)

In [8]:
print(df_granted.columns.tolist())

['application_number', 'event_code', 'recorded_date', 'event_desc', 'event_order', 'examiner_full_name', 'examiner_art_unit', 'patent_number', 'filing_year', 'patent_id', 'patent_type', 'num_claims', 'inventor_count']


In [9]:
print(df_granted[['patent_type', 'num_claims', 'inventor_count']].isna().sum())

patent_type        154
num_claims         154
inventor_count    1139
dtype: int64


In [10]:
# 결측값 있는 application_number 찾기
null_cases = df_granted[df_granted[['patent_type', 'num_claims', 'inventor_count']].isna().any(axis=1)]['application_number'].unique()

# 해당 케이스 전체 제거
df_granted = df_granted[~df_granted['application_number'].isin(null_cases)].reset_index(drop=True)

In [11]:
df_granted = df_granted.drop_duplicates().reset_index(drop=True)
print(df_granted.shape)

(95177963, 13)


In [12]:
df_granted = df_granted.drop(columns='event_order')
df_granted['recorded_date'] = pd.to_datetime(df_granted['recorded_date'], format='mixed')
df_granted = df_granted.sort_values(['application_number', 'recorded_date']).reset_index(drop=True)
df_granted['event_order'] = df_granted.groupby('application_number').cumcount() + 1
print(df_granted[['application_number', 'recorded_date', 'event_desc', 'event_order']].head(10))

   application_number recorded_date                                event_desc  \
0             9786529    2010-10-27                            Filing Receipt   
1             9786529    2010-10-27         Notice of DO/EO Acceptance Mailed   
2             9786529    2011-01-18                      Mail Pre-Exam Notice   
3             9786529    2011-01-18          Application Dispatched from OIPE   
4             9786529    2011-01-18                Application Return TO OIPE   
5             9786529    2011-01-31          Case Docketed to Examiner in GAU   
6             9786529    2011-02-09       Non-Compliant Preliminary Amendment   
7             9786529    2011-02-10  Mail Non-Compliant Preliminary Amendment   
8             9786529    2011-02-17          Case Docketed to Examiner in GAU   
9             9786529    2011-03-08                     Preliminary Amendment   

   event_order  
0            1  
1            2  
2            3  
3            4  
4            5  
5     

In [13]:
df_granted['patent_type'].value_counts()

patent_type
utility    90250594
design      4555476
reissue      212091
plant        159802
Name: count, dtype: int64

In [14]:
df_granted = df_granted[df_granted['patent_type'] == 'utility'].reset_index(drop=True)

In [15]:
# year_month 컬럼 생성
df_granted['year_month'] = df_granted['recorded_date'].dt.to_period('M').astype(str)

df_granted['examiner_art_unit'] = df_granted['examiner_art_unit'].astype(str).str.strip()
df_granted['examiner_full_name'] = df_granted['examiner_full_name'].astype(str).str.strip()

# examiner workload merge
df_granted = df_granted.merge(
    examiner_workload,
    on=['examiner_full_name', 'year_month'],
    how='left'
)

# art unit backlog merge
df_granted = df_granted.merge(
    artunit_backlog,
    on=['examiner_art_unit', 'year_month'],
    how='left'
)

print(df_granted[['examiner_workload', 'artunit_backlog']].isna().sum())

examiner_workload    5226251
artunit_backlog        52442
dtype: int64


In [16]:
df_granted['examiner_art_unit'].isna().sum()

np.int64(0)

In [17]:
print(df_granted[['examiner_workload', 'artunit_backlog']].isna().mean().round(3))
print(f"전체 rows: {len(df_granted):,}")

examiner_workload    0.036
artunit_backlog      0.000
dtype: float64
전체 rows: 143,440,250


In [19]:
null_cases = df_granted[df_granted['examiner_workload'].isna()]['application_number'].unique()
df_granted = df_granted[~df_granted['application_number'].isin(null_cases)].reset_index(drop=True)

print(f"남은 케이스: {df_granted['application_number'].nunique():,}")
print(f"남은 rows: {len(df_granted):,}")

남은 케이스: 1,454,420
남은 rows: 102,397,740


In [20]:
null_cases = df_granted[df_granted['artunit_backlog'].isna()]['application_number'].unique()
df_granted = df_granted[~df_granted['application_number'].isin(null_cases)].reset_index(drop=True)

print(f"남은 케이스: {df_granted['application_number'].nunique():,}")
print(f"남은 rows: {len(df_granted):,}")

남은 케이스: 1,452,372
남은 rows: 102,304,746


# train-test split

In [21]:
grant_date = (df_granted.groupby('application_number')['recorded_date']
              .max().rename('grant_date'))
df_granted = df_granted.merge(grant_date, on='application_number')
df_granted = df_granted.drop(columns='filing_year')

df_granted['remaining_days'] = (df_granted['grant_date'] - df_granted['recorded_date']).dt.days

first_date = (df_granted.groupby('application_number')['recorded_date']
              .min().rename('first_date'))
df_granted = df_granted.merge(first_date, on='application_number')
df_granted['elapsed_days'] = (df_granted['recorded_date'] - df_granted['first_date']).dt.days

prefix = df_granted[df_granted['remaining_days'] > 0].copy()
prefix = prefix[['application_number', 'event_desc', 'event_order',
                  'elapsed_days', 'remaining_days',
                  'examiner_art_unit', 'examiner_full_name',
                  'patent_type', 'num_claims', 'inventor_count',
                  'examiner_workload', 'artunit_backlog']].copy()

# train/test split
cases = df_granted[['application_number', 'first_date']].drop_duplicates()
train_cases = set(cases[cases['first_date'] < '2014-01-01']['application_number'])
test_cases  = set(cases[cases['first_date'] >= '2014-01-01']['application_number'])

print(f"train: {len(train_cases):,} cases")
print(f"test : {len(test_cases):,} cases")

train: 982,721 cases
test : 469,651 cases


In [22]:
# prefix 분리
train_prefix = prefix[prefix['application_number'].isin(train_cases)]
test_prefix  = prefix[prefix['application_number'].isin(test_cases)]

print(f"\ntrain rows: {len(train_prefix):,}")
print(f"test rows : {len(test_prefix):,}")


train rows: 66,724,046
test rows : 33,087,963


In [23]:
# label encoding
le_desc = LabelEncoder()
le_art  = LabelEncoder()

le_desc.fit(train_prefix['event_desc'])
le_art.fit(train_prefix['examiner_art_unit'].astype(str))

# train
X_train_1 = train_prefix[['event_order', 'elapsed_days']].copy()
X_train_1['event_desc']        = le_desc.transform(train_prefix['event_desc'])

X_train_2 = X_train_1.copy()
X_train_2['examiner_art_unit'] = le_art.transform(train_prefix['examiner_art_unit'].astype(str))
X_train_2['num_claims']        = train_prefix['num_claims'].values
X_train_2['inventor_count']    = train_prefix['inventor_count'].values

# test (unknown 처리 포함)
test_desc = test_prefix['event_desc'].where(test_prefix['event_desc'].isin(le_desc.classes_), le_desc.classes_[0])
test_art  = test_prefix['examiner_art_unit'].astype(str).where(test_prefix['examiner_art_unit'].astype(str).isin(le_art.classes_), le_art.classes_[0])

X_test_1 = test_prefix[['event_order', 'elapsed_days']].copy()
X_test_1['event_desc']        = le_desc.transform(test_desc)

X_test_2 = X_test_1.copy()
X_test_2['examiner_art_unit'] = le_art.transform(test_art)
X_test_2['num_claims']        = test_prefix['num_claims'].values
X_test_2['inventor_count']    = test_prefix['inventor_count'].values

y_train = train_prefix['remaining_days'].values
y_test  = test_prefix['remaining_days'].values

print(X_train_1.shape, X_train_2.shape)
print(X_test_1.shape,  X_test_2.shape)

(66724046, 3) (66724046, 6)
(33087963, 3) (33087963, 6)


## 베이스라인 평균부터

In [24]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

y_mean = y_train.mean()
mae_baseline  = mean_absolute_error(y_test, np.full(len(y_test), y_mean))
rmse_baseline = np.sqrt(mean_squared_error(y_test, np.full(len(y_test), y_mean)))

print(f"평균 예측 baseline MAE  : {mae_baseline:.1f} days")
print(f"평균 예측 baseline RMSE : {rmse_baseline:.1f} days")
print(f"y_train 평균: {y_train.mean():.1f} days")
print(f"y_train 중앙값: {np.median(y_train):.1f} days")

평균 예측 baseline MAE  : 305.7 days
평균 예측 baseline RMSE : 384.1 days
y_train 평균: 431.7 days
y_train 중앙값: 260.0 days


# Approach 1

## LR

In [25]:
from sklearn.linear_model import LinearRegression

lr_1 = LinearRegression()
lr_1.fit(X_train_1, y_train)

y_pred_lr_1 = lr_1.predict(X_test_1)
mae_lr_1  = mean_absolute_error(y_test, y_pred_lr_1)
rmse_lr_1 = np.sqrt(mean_squared_error(y_test, y_pred_lr_1))

print(f"[Approach 1] Linear Regression MAE  : {mae_lr_1:.1f} days")
print(f"[Approach 1] Linear Regression RMSE : {rmse_lr_1:.1f} days")

[Approach 1] Linear Regression MAE  : 291.2 days
[Approach 1] Linear Regression RMSE : 363.9 days


## XGBoost

In [26]:
# !pip install xgboost

In [27]:
import xgboost as xgb

xgb_1 = xgb.XGBRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=6,
    random_state=42,
    tree_method='hist',
    device='cuda'
)

xgb_1.fit(X_train_1, y_train)

y_pred_xgb_1 = xgb_1.predict(X_test_1)
mae_xgb_1  = mean_absolute_error(y_test, y_pred_xgb_1)
rmse_xgb_1 = np.sqrt(mean_squared_error(y_test, y_pred_xgb_1))

print(f"[Approach 1] XGBoost MAE  : {mae_xgb_1:.1f} days")
print(f"[Approach 1] XGBoost RMSE : {rmse_xgb_1:.1f} days")

c:\Users\DME Lab 2\AppData\Local\Programs\Python\Python313\Lib\site-packages\xgboost\core.py:751: UserWarning: [12:51:59] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


[Approach 1] XGBoost MAE  : 214.8 days
[Approach 1] XGBoost RMSE : 310.0 days


# Approach 2

## LR

In [28]:
lr_2 = LinearRegression()
lr_2.fit(X_train_2, y_train)

y_pred_lr_2 = lr_2.predict(X_test_2)
mae_lr_2  = mean_absolute_error(y_test, y_pred_lr_2)
rmse_lr_2 = np.sqrt(mean_squared_error(y_test, y_pred_lr_2))

print(f"[Approach 2] Linear Regression MAE  : {mae_lr_2:.1f} days")
print(f"[Approach 2] Linear Regression RMSE : {rmse_lr_2:.1f} days")

[Approach 2] Linear Regression MAE  : 289.6 days
[Approach 2] Linear Regression RMSE : 362.4 days


# XGBoost

In [29]:
xgb_2 = xgb.XGBRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=6,
    random_state=42,
    tree_method='hist',
    device='cuda'
)

xgb_2.fit(X_train_2, y_train)

y_pred_xgb_2 = xgb_2.predict(X_test_2)
mae_xgb_2  = mean_absolute_error(y_test, y_pred_xgb_2)
rmse_xgb_2 = np.sqrt(mean_squared_error(y_test, y_pred_xgb_2))

print(f"[Approach 2] XGBoost MAE  : {mae_xgb_2:.1f} days")
print(f"[Approach 2] XGBoost RMSE : {rmse_xgb_2:.1f} days")

[Approach 2] XGBoost MAE  : 211.4 days
[Approach 2] XGBoost RMSE : 302.2 days


# Approach 3

In [30]:
# Approach 3: X_train_2에 inter-case features 추가
X_train_3 = X_train_2.copy()
X_train_3['examiner_workload'] = train_prefix['examiner_workload'].values
X_train_3['artunit_backlog']   = train_prefix['artunit_backlog'].values

X_test_3 = X_test_2.copy()
X_test_3['examiner_workload'] = test_prefix['examiner_workload'].values
X_test_3['artunit_backlog']   = test_prefix['artunit_backlog'].values

## LR

In [31]:
print(X_train_3.isna().sum())
print(X_test_3.isna().sum())

event_order          0
elapsed_days         0
event_desc           0
examiner_art_unit    0
num_claims           0
inventor_count       0
examiner_workload    0
artunit_backlog      0
dtype: int64
event_order          0
elapsed_days         0
event_desc           0
examiner_art_unit    0
num_claims           0
inventor_count       0
examiner_workload    0
artunit_backlog      0
dtype: int64


In [32]:
lr_3 = LinearRegression()
lr_3.fit(X_train_3, y_train)
y_pred_lr_3 = lr_3.predict(X_test_3)
mae_lr_3  = mean_absolute_error(y_test, y_pred_lr_3)
rmse_lr_3 = np.sqrt(mean_squared_error(y_test, y_pred_lr_3))
print(f"[Approach 3] Linear Regression MAE  : {mae_lr_3:.1f} days")
print(f"[Approach 3] Linear Regression RMSE : {rmse_lr_3:.1f} days")

[Approach 3] Linear Regression MAE  : 277.7 days
[Approach 3] Linear Regression RMSE : 351.6 days


## XGBoost

In [33]:
xgb_3 = xgb.XGBRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=6,
    random_state=42,
    tree_method='hist',
    device='cuda'
)
xgb_3.fit(X_train_3, y_train)
y_pred_xgb_3 = xgb_3.predict(X_test_3)
mae_xgb_3  = mean_absolute_error(y_test, y_pred_xgb_3)
rmse_xgb_3 = np.sqrt(mean_squared_error(y_test, y_pred_xgb_3))
print(f"[Approach 3] XGBoost MAE  : {mae_xgb_3:.1f} days")
print(f"[Approach 3] XGBoost RMSE : {rmse_xgb_3:.1f} days")

[Approach 3] XGBoost MAE  : 204.3 days
[Approach 3] XGBoost RMSE : 294.2 days


# 최종결과

In [ ]:
results = pd.DataFrame({
    'Model': ['Baseline (Mean)', 'Linear Regression', 'XGBoost'],
    'Approach 1 MAE': [mae_baseline, mae_lr_1, mae_xgb_1],
    'Approach 1 RMSE': [rmse_baseline, rmse_lr_1, rmse_xgb_1],
    'Approach 2 MAE': [None, mae_lr_2, mae_xgb_2],
    'Approach 2 RMSE': [None, rmse_lr_2, rmse_xgb_2],
    'Approach 3 MAE': [None, mae_lr_3, mae_xgb_3],
    'Approach 3 RMSE': [None, rmse_lr_3, rmse_xgb_3],
})

print(results.to_string(index=False))

            Model  A1 MAE  A1 RMSE A2 MAE A2 RMSE A3 MAE A3 RMSE
  Baseline (Mean)   305.7    384.1      -       -      -       -
Linear Regression   291.2    363.9  289.6   362.4  277.7   351.6
          XGBoost   214.8    310.0  211.4   302.2  204.3   294.2


In [36]:
results = pd.DataFrame({
    'Model': ['Baseline (Mean)', 'Linear Regression', 'XGBoost'],
    'A1 MAE': [round(mae_baseline,1), round(mae_lr_1,1), round(mae_xgb_1,1)],
    'A1 RMSE': [round(rmse_baseline,1), round(rmse_lr_1,1), round(rmse_xgb_1,1)],
    'A2 MAE': ['-', round(mae_lr_2,1), round(mae_xgb_2,1)],
    'A2 RMSE': ['-', round(rmse_lr_2,1), round(rmse_xgb_2,1)],
    'A3 MAE': ['-', round(mae_lr_3,1), round(mae_xgb_3,1)],
    'A3 RMSE': ['-', round(rmse_lr_3,1), round(rmse_xgb_3,1)],
})

print(results.to_string(index=False))

            Model  A1 MAE  A1 RMSE A2 MAE A2 RMSE A3 MAE A3 RMSE
  Baseline (Mean)   305.7    384.1      -       -      -       -
Linear Regression   291.2    363.9  289.6   362.4  277.7   351.6
          XGBoost   214.8    310.0  211.4   302.2  204.3   294.2


In [37]:
from IPython.display import display
import pandas as pd

results = pd.DataFrame({
    'Model': ['Baseline (Mean)', 'Linear Regression', 'XGBoost'],
    'A1 MAE': [round(mae_baseline,1), round(mae_lr_1,1), round(mae_xgb_1,1)],
    'A1 RMSE': [round(rmse_baseline,1), round(rmse_lr_1,1), round(rmse_xgb_1,1)],
    'A2 MAE': ['-', round(mae_lr_2,1), round(mae_xgb_2,1)],
    'A2 RMSE': ['-', round(rmse_lr_2,1), round(rmse_xgb_2,1)],
    'A3 MAE': ['-', round(mae_lr_3,1), round(mae_xgb_3,1)],
    'A3 RMSE': ['-', round(rmse_lr_3,1), round(rmse_xgb_3,1)],
}).set_index('Model')

display(results)

,A1 MAE,A1 RMSE,A2 MAE,A2 RMSE,A3 MAE,A3 RMSE
Model,,,,,,
Baseline (Mean),305.7,384.1,-,-,-,-
Linear Regression,291.2,363.9,289.6,362.4,277.7,351.6
XGBoost,214.8,310.0,211.4,302.2,204.3,294.2
